# Imports

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno

# Current Registry

In [2]:
registry_df = pd.read_csv("current_rental_registrations_251001.csv")

In [3]:
registry_df["formatted_address"] = registry_df["RegisteredAddress"].str.replace(r"\s+,", ",", regex=True)

In [4]:
registry_df["formatted_address"] = registry_df["formatted_address"].astype(str).str.upper().str.strip()

In [5]:
# Remove unit numbers before the first comma (e.g., " AVE 5," → " AVE,")
registry_df["formatted_address"] = registry_df["formatted_address"].str.replace(
    r"\s+\d+(?=,)", "", regex=True
).str.strip()


In [6]:
import re

def clean_units(address):
    # Match pattern: [street] [unit], [city], MA [ZIP]
    match = re.match(r"^(.*\b(?:ST|AV|AVE|RD|BLVD|PL|CT|DR|TER|WAY|LN|SQ|TE|CIR|PKWY|PLZ|HWY))\s+[A-Z0-9\-]+, (.+?, MA \d{5})$", address)
    if match:
        return f"{match.group(1)}, {match.group(2)}"
    return address

registry_df["formatted_address"] = registry_df["formatted_address"].apply(clean_units)


In [7]:
registry_df['formatted_address'].sample(30)

28735                  51 ETNA ST, BRIGHTON, MA 02135
33332              31 HENDRY ST, DORCHESTER, MA 02122
13150               2 PRESIDENT TE, ALLSTON, MA 02134
5453     23-25 SAINT BRENDAN RD, DORCHESTER, MA 02124
37590           60-62 MAPLETON ST, BRIGHTON, MA 02135
26758          265-275 DARTMOUTH ST, BOSTON, MA 02116
16467        1980 DORCHESTER AV, DORCHESTER, MA 02124
36081             14 LAFIELD ST, DORCHESTER, MA 02122
33356                40 HEREFORD ST, BOSTON, MA 02115
4012               32 REEDSDALE ST, ALLSTON, MA 02134
39459                4 MORELAND ST, ROXBURY, MA 02119
33969        153-155A HOWARD AV, DORCHESTER, MA 02125
35854                32 KING ST, DORCHESTER, MA 02122
35249              24 JEROME ST, DORCHESTER, MA 02125
18393             285 CENTRE ST, DORCHESTER, MA 02122
19755        247 CHESTNUT HILL AV, BRIGHTON, MA 02135
1855                    131 PARK DR, BOSTON, MA 02215
14267                161 KELTON ST, ALLSTON, MA 02134
38984            17 MICHIGAN

# Assessor Dataset

In [8]:
assessment_df = pd.read_csv("fy2025-property-assessment-data_12_30_2024.csv", dtype={21: str}, low_memory=False)

In [9]:
def combine_street_numbers(row):
    try:
        st_num = str(int(float(row['ST_NUM']))) if pd.notnull(row['ST_NUM']) else ""
        st_num2 = str(int(float(row['ST_NUM2']))) if pd.notnull(row['ST_NUM2']) else ""
        return f"{st_num}-{st_num2}" if st_num and st_num2 else st_num
    except:
        return ""

assessment_df["ST_NUM_COMBINED"] = assessment_df.apply(combine_street_numbers, axis=1)


In [10]:
assessment_df["ZIP_CODE_CLEAN"] = assessment_df["ZIP_CODE"].astype(str).str.extract(r'(\d+)')[0].str.zfill(5)

assessment_df["ST_NAME_CLEAN"] = assessment_df["ST_NAME"].astype(str).str.replace(r'\bAVE\.', 'AV', regex=True)

assessment_df["formatted_address"] = (
    assessment_df["ST_NUM_COMBINED"].str.strip() + " " +
    assessment_df["ST_NAME_CLEAN"].astype(str).str.strip() + ", " +
    assessment_df["CITY"].astype(str).str.strip() + ", MA " +
    assessment_df["ZIP_CODE_CLEAN"]
).str.upper()


In [11]:
assessment_df["formatted_address"].dropna().unique()[:10]

array(['104 PUTNAM ST, EAST BOSTON, MA 02128',
       '197 LEXINGTON ST, EAST BOSTON, MA 02128',
       '199 LEXINGTON ST, EAST BOSTON, MA 02128',
       '201 LEXINGTON ST, EAST BOSTON, MA 02128',
       '203 LEXINGTON ST, EAST BOSTON, MA 02128',
       '205-207 LEXINGTON ST, EAST BOSTON, MA 02128',
       '209-211 LEXINGTON ST, EAST BOSTON, MA 02128',
       '213 LEXINGTON ST, EAST BOSTON, MA 02128',
       '215 LEXINGTON ST, EAST BOSTON, MA 02128',
       '217 LEXINGTON ST, EAST BOSTON, MA 02128'], dtype=object)

In [12]:
# Show side-by-side original and formatted addresses from a sample
assessment_df[["ST_NUM", "ST_NUM2", "ST_NAME", "CITY", "ZIP_CODE", "formatted_address"]].sample(50)


,ST_NUM,ST_NUM2,ST_NAME,CITY,ZIP_CODE,formatted_address
138536,NaN,NaN,BRAEBURN RD,HYDE PARK,2136.0,"BRAEBURN RD, HYDE PARK, MA 02136"
149822,72.0,NaN,COHASSET ST,ROSLINDALE,2131.0,"72 COHASSET ST, ROSLINDALE, MA 02131"
144797,26.0,NaN,POND ST,JAMAICA PLAIN,2130.0,"26 POND ST, JAMAICA PLAIN, MA 02130"
95679,95.0,NaN,MUNROE ST,ROXBURY,2119.0,"95 MUNROE ST, ROXBURY, MA 02119"
5359,NaN,NaN,Geneva ST,EAST BOSTON,2128.0,"GENEVA ST, EAST BOSTON, MA 02128"
43662,20.0,NaN,Gray ST,BOSTON,2116.0,"20 GRAY ST, BOSTON, MA 02116"
38780,25.0,NaN,ST GERMAIN ST,BOSTON,2115.0,"25 ST GERMAIN ST, BOSTON, MA 02115"
182385,21.0,NaN,ANSELM TE,BRIGHTON,2135.0,"21 ANSELM TE, BRIGHTON, MA 02135"
75293,117.0,NaN,Buttonwood ST,DORCHESTER,2125.0,"117 BUTTONWOOD ST, DORCHESTER, MA 02125"
36503,178.0,NaN,W Canton ST,BOSTON,2116.0,"178 W CANTON ST, BOSTON, MA 02116"


# Current vs Assessor

In [13]:
# Compare formatted FY2025 addresses to registry addresses
unregistered_props = assessment_df[~assessment_df["formatted_address"].isin(registry_df["formatted_address"])]

# Count how many are unregistered
unregistered_count = unregistered_props.shape[0]

# Total properties in FY2025
total_properties = assessment_df.shape[0]

# Calculate percentage unregistered
unregistered_pct = (unregistered_count / total_properties) * 100

unregistered_count, total_properties, round(unregistered_pct, 2)

(134491, 183445, 73.31)

In [27]:
registered_properties_from_assessment = assessment_df[
    assessment_df["formatted_address"].isin(registry_df["formatted_address"])
]

# Show first 10 matching addresses
registered_properties_from_assessment["formatted_address"].sample(30)

50753                103 BEACON ST, BOSTON, MA 02116
156893            79 ROBERT ST, ROSLINDALE, MA 02131
59090           122 TUDOR ST, SOUTH BOSTON, MA 02127
134198         24-26 SAFFORD ST, HYDE PARK, MA 02136
7376            64 EVERETT ST, EAST BOSTON, MA 02128
47378               78 PHILLIPS ST, BOSTON, MA 02114
122556         39 FAIRMOUNT ST, DORCHESTER, MA 02124
99877               21 TAFT ST, DORCHESTER, MA 02125
182527              16 GERALD RD, BRIGHTON, MA 02135
167530            56 PARK VALE AV, ALLSTON, MA 02134
11593             299 MAIN ST, CHARLESTOWN, MA 02129
3728           171 TRENTON ST, EAST BOSTON, MA 02128
118490           7-9 BECKET ST, DORCHESTER, MA 02124
45517          160 COMMONWEALTH AV, BOSTON, MA 02116
26743                  80 BROAD ST, BOSTON, MA 02110
65107              505 CONGRESS ST, BOSTON, MA 02210
103899           10 NORWELL ST, DORCHESTER, MA 02121
17033           47 HARVARD ST, CHARLESTOWN, MA 02129
42636       1 CHARLES STREET SOUTH, BOSTON, MA

# 311 Service Request

In [15]:
service_df = pd.read_csv("dff4d804-5031-443a-8409-8344efd0e5c8.csv", low_memory=False)

In [24]:
# Known city/neighborhood names to catch multi-word places like "SOUTH BOSTON", "JAMAICA PLAIN"
boston_neighborhoods = [
    "SOUTH BOSTON", "EAST BOSTON", "JAMAICA PLAIN", "MATTAPAN", "ROXBURY", 
    "BRIGHTON", "CHARLESTOWN", "HYDE PARK", "DORCHESTER", "WEST ROXBURY", 
    "ALLSTON", "ROSLINDALE", "BACK BAY", "FENWAY", "MISSION HILL", "NORTH END",
    "SOUTH END", "CHINATOWN"
]

def format_location(location):
    try:
        parts = location.strip().split()
        if len(parts) < 4:
            return location.upper()

        zip_code = parts[-1]
        state = parts[-2]

        # Try 2-word city names first
        possible_city = " ".join(parts[-4:-2]).upper()
        if possible_city in boston_neighborhoods:
            city = possible_city
            street = " ".join(parts[:-4])
        else:
            # Fall back to 1-word city names
            city = parts[-3].upper()
            street = " ".join(parts[:-3])

        return f"{street}, {city}, {state} {zip_code}".upper()
    except:
        return ""
service_df["formatted_address"] = service_df["location"].astype(str).apply(format_location)

# Replace AVE (or AVE.) with AV in 311 formatted addresses
service_df["formatted_address"] = service_df["formatted_address"].str.replace(
    r"\bAVE\.?\b", "AV", regex=True
)


service_df["formatted_address"].sample(30)

78122                     23 WOODBINE ST, ROXBURY, MA 02119
264422               132-148 MAIN ST, CHARLESTOWN, MA 02129
63958     INTERSECTION OF MOTHER JULIA RD & DORCHESTER, ...
244592                    211 WESTERN AV, ALLSTON, MA 02134
166700                   32 ROCKNE AV, DORCHESTER, MA 02124
57740           395-397 MASSACHUSETTS AV, ROXBURY, MA 02118
192412                   9 GATES ST, SOUTH BOSTON, MA 02127
199955             309 CHESTNUT AV, JAMAICA PLAIN, MA 02130
133646                   189 BEECH ST, ROSLINDALE, MA 02131
265459                 71 CROCKETT AV, DORCHESTER, MA 02124
279357                     139 TREMONT ST, BOSTON, MA 02108
268734                     645 RIVER ST, MATTAPAN, MA 02126
235439                      59 BURBANK ST, BOSTON, MA 02115
75385                    60 NELSON ST, DORCHESTER, MA 02124
27280                                                      
104368               180 W FIFTH ST, SOUTH BOSTON, MA 02127
48591                      9 S RUSSELL S

# Compare Current vs 311 

In [25]:
unregistered_service_requests = service_df[~service_df["formatted_address"].isin(registry_df["formatted_address"])]

unregistered_service_count = unregistered_service_requests.shape[0]
total_service_requests = service_df.shape[0]
unregistered_service_pct = (unregistered_service_count / total_service_requests) * 100

unregistered_service_count, total_service_requests, round(unregistered_service_pct, 2)


(229914, 282836, 81.29)

In [26]:
registered_service_requests = service_df[service_df["formatted_address"].isin(registry_df["formatted_address"])]

registered_service_requests["formatted_address"].sample(30)

223428                7 WARREN AV, BOSTON, MA 02116
164660      348 E EIGHTH ST, SOUTH BOSTON, MA 02127
97606          76 PETERBOROUGH ST, BOSTON, MA 02215
218900      484-486 COMMERCIAL ST, BOSTON, MA 02109
34842           70 MARLBOROUGH ST, BOSTON, MA 02116
112880    157 DORCHESTER ST, SOUTH BOSTON, MA 02127
258759              49 HANCOCK ST, BOSTON, MA 02114
220250       48 E SPRINGFIELD ST, ROXBURY, MA 02118
87777         1640 WASHINGTON ST, ROXBURY, MA 02118
50349         497 COMMONWEALTH AV, BOSTON, MA 02215
105908       74 HILLSIDE ST, MISSION HILL, MA 02120
168329         87 BELGRADE AV, ROSLINDALE, MA 02131
3019           46 WILDWOOD ST, DORCHESTER, MA 02124
114811           1282 BOYLSTON ST, BOSTON, MA 02215
9792       1615 COMMONWEALTH AV, BRIGHTON, MA 02135
95477              40 ANDERSON ST, BOSTON, MA 02114
49702      1730 COMMONWEALTH AV, BRIGHTON, MA 02135
273115        109 MOUNT VERNON ST, BOSTON, MA 02114
82858        151 W SIXTH ST, SOUTH BOSTON, MA 02127
266262      

# Unregistered list